# Phase 1: Tree Ensembles & Boosting (Air Quality Data)

Dataset: Delhi Air Quality (chronological split — no shuffle)  
Splits: **70% Train | 15% Val | 15% Test**  
- Q1.1–1.3: **Binary classification** — label = 1 if PM2.5 > 200, else 0  
- Q1.4: **Regression** — continuous PM2.5 target

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Loading & Chronological Splits

In [ ]:
CSV_PATH = '../Datasets/delhi_aqi_dataset/delhi_aqi.csv'
df = pd.read_csv(CSV_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)   # chronological order
df = df.dropna().reset_index(drop=True)

FEATURE_COLS = ['co', 'no', 'no2', 'o3', 'so2', 'pm10', 'nh3']  # exclude target
TARGET_COL   = 'pm2_5'

X     = df[FEATURE_COLS].values
y_reg = df[TARGET_COL].values          # continuous — Q1.4
y_cls = (y_reg > 200).astype(int)      # binary    — Q1.1–1.3

# Chronological (no shuffle) split — do not shuffle to avoid temporal leakage
n       = len(X)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

X_train, X_val, X_test         = X[:n_train], X[n_train:n_train+n_val], X[n_train+n_val:]
y_cls_train, y_cls_val, y_cls_test = y_cls[:n_train], y_cls[n_train:n_train+n_val], y_cls[n_train+n_val:]
y_reg_train, y_reg_val, y_reg_test = y_reg[:n_train], y_reg[n_train:n_train+n_val], y_reg[n_train+n_val:]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Class balance — 0:{(y_cls_train==0).sum()}  1:{(y_cls_train==1).sum()}")

---
## Shared Utilities: Decision Tree Primitives

In [ ]:
# ── Gini impurity ─────────────────────────────────────────────────────────────
def gini_impurity(y):
    """Gini impurity = 1 - Σ p_k^2   (binary labels)."""
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    return 1.0 - np.sum(p ** 2)


def gini_gain_best(feature_vals, labels):
    """
    Find the threshold on a continuous feature that maximises Gini gain.
    Iterates over all unique mid-point thresholds (vectorised).
    Returns (best_threshold, best_gain).
    """
    sorted_unique = np.sort(np.unique(feature_vals))
    thresholds    = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0
    n             = len(labels)
    gini_parent   = gini_impurity(labels)

    best_gain, best_thresh = -np.inf, None
    for t in thresholds:
        left  = labels[feature_vals <= t]
        right = labels[feature_vals > t]
        if len(left) == 0 or len(right) == 0:
            continue
        gain = gini_parent - (len(left)/n * gini_impurity(left)
                              + len(right)/n * gini_impurity(right))
        if gain > best_gain:
            best_gain, best_thresh = gain, t
    return best_thresh, best_gain


# ── Decision tree node ────────────────────────────────────────────────────────
class Node:
    __slots__ = ['feat_idx', 'threshold', 'left', 'right', 'value']

    def __init__(self, feat_idx=None, threshold=None, left=None, right=None, value=None):
        self.feat_idx  = feat_idx
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value  # leaf: majority class or mean (regression)


def _best_split_cls(X, y, feat_indices):
    """Best (feature, threshold) among feat_indices for classification (Gini)."""
    bg, bf, bt = -np.inf, None, None
    for fi in feat_indices:
        t, g = gini_gain_best(X[:, fi], y)
        if t is not None and g > bg:
            bg, bf, bt = g, fi, t
    return bf, bt


def _best_split_reg(X, y, feat_indices):
    """Best (feature, threshold) among feat_indices for regression (MSE)."""
    best_mse, bf, bt = np.inf, None, None
    for fi in feat_indices:
        vals = X[:, fi]
        sv   = np.sort(np.unique(vals))
        for t in (sv[:-1] + sv[1:]) / 2.0:
            lm = vals <= t; rm = ~lm
            if lm.sum() == 0 or rm.sum() == 0:
                continue
            mse = np.var(y[lm]) * lm.sum() + np.var(y[rm]) * rm.sum()
            if mse < best_mse:
                best_mse, bf, bt = mse, fi, t
    return bf, bt


def build_tree_cls(X, y, max_depth, n_feats, depth=0):
    """Build a classification tree (Gini criterion)."""
    if depth >= max_depth or len(np.unique(y)) == 1 or len(y) < 2:
        return Node(value=int(np.bincount(y).argmax()))
    fi = np.random.choice(X.shape[1], n_feats, replace=False)
    f, t = _best_split_cls(X, y, fi)
    if f is None:
        return Node(value=int(np.bincount(y).argmax()))
    lm = X[:, f] <= t
    node = Node(feat_idx=f, threshold=t)
    node.left  = build_tree_cls(X[lm],  y[lm],  max_depth, n_feats, depth+1)
    node.right = build_tree_cls(X[~lm], y[~lm], max_depth, n_feats, depth+1)
    return node


def build_tree_reg(X, y, max_depth, depth=0):
    """Build a regression tree (MSE criterion)."""
    if depth >= max_depth or len(y) < 2:
        return Node(value=np.mean(y))
    fi = np.arange(X.shape[1])  # use all features for regression stumps
    f, t = _best_split_reg(X, y, fi)
    if f is None:
        return Node(value=np.mean(y))
    lm = X[:, f] <= t
    node = Node(feat_idx=f, threshold=t)
    node.left  = build_tree_reg(X[lm],  y[lm],  max_depth, depth+1)
    node.right = build_tree_reg(X[~lm], y[~lm], max_depth, depth+1)
    return node


def predict_tree(node, X):
    """Vectorised tree prediction over a batch X."""
    n = len(X)
    out = np.empty(n)
    # iterative traversal with index masks
    stack = [(node, np.arange(n))]
    while stack:
        cur, idx = stack.pop()
        if cur.value is not None:
            out[idx] = cur.value
        else:
            go_left  = X[idx, cur.feat_idx] <= cur.threshold
            stack.append((cur.left,  idx[go_left]))
            stack.append((cur.right, idx[~go_left]))
    return out


def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()

---
## Q1.1 — Information Gain (Gini Impurity Reduction) from Scratch

Iterates over the **`no2`** feature  
*(Note: the downloaded dataset has no temperature column; `no2` is used as the proxy continuous meteorological-like feature.)*

Reports the optimal split threshold that maximises Gini gain at the root node.

In [ ]:
NO2_IDX = FEATURE_COLS.index('no2')
feature_vals = X_train[:, NO2_IDX]

optimal_threshold, optimal_gain = gini_gain_best(feature_vals, y_cls_train)

print(f"Feature           : no2 (index {NO2_IDX})")
print(f"Optimal threshold : {optimal_threshold:.4f}")
print(f"Gini gain         : {optimal_gain:.6f}")

---
## Q1.2 — Random Forest (from Scratch)

- Bootstrap row sampling (bagging)
- Feature bagging: each tree uses √(n_features) random features at every split
- Depth-limited trees
- Majority vote ensemble prediction

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=20, max_depth=5, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.random_state = random_state
        self.trees        = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        n_feats   = max(1, int(np.sqrt(X.shape[1])))  # √p feature subspaces
        self.trees = []
        for i in range(self.n_estimators):
            # Bootstrap sample
            idx  = np.random.choice(n_samples, n_samples, replace=True)
            tree = build_tree_cls(X[idx], y[idx], self.max_depth, n_feats)
            self.trees.append(tree)
            print(f"  tree {i+1}/{self.n_estimators} done", flush=True)
        return self

    def predict(self, X):
        # Majority vote
        votes = np.stack([predict_tree(t, X) for t in self.trees])  # (n_trees, n_samples)
        return (votes.mean(axis=0) >= 0.5).astype(int)


rf = RandomForest(n_estimators=20, max_depth=5, random_state=42)
rf.fit(X_train, y_cls_train)

val_acc  = accuracy(y_cls_val,  rf.predict(X_val))
test_acc = accuracy(y_cls_test, rf.predict(X_test))
print(f"\nRandom Forest — Val Accuracy : {val_acc:.4f}")
print(f"Random Forest — Test Accuracy: {test_acc:.4f}")

---
## Q1.3 — AdaBoost Sample Reweighting (from Scratch)

Implements **AdaBoost.M1** with decision stumps (depth = 1).  
Tracks sample weights of the **5 most consistently misclassified samples** over 50 iterations.  
Labels mapped to {−1, +1} internally.

In [ ]:
class AdaBoost:
    """
    AdaBoost.M1 with decision stumps (depth=1).
    Update rule:  w_i ← w_i · exp(−α · y_i · h(x_i)),  then normalise.
    """
    def __init__(self, n_estimators=50, stump_depth=1, random_state=42):
        self.n_estimators = n_estimators
        self.stump_depth  = stump_depth
        self.random_state = random_state
        self.alphas, self.stumps = [], []

    def fit(self, X, y, track_indices=None):
        """
        Fit AdaBoost.
        track_indices: sample indices to record weights for each iteration.
        Returns weight_history array (n_estimators × len(track_indices)) if given.
        """
        np.random.seed(self.random_state)
        n    = len(y)
        y_pm = np.where(y == 1, 1.0, -1.0)   # {-1, +1} encoding
        w    = np.ones(n) / n                 # uniform initial weights

        weight_history = []

        for _ in range(self.n_estimators):
            if track_indices is not None:
                weight_history.append(w[track_indices].copy())

            # Weighted bootstrap then train stump
            idx   = np.random.choice(n, n, replace=True, p=w / w.sum())
            stump = build_tree_cls(X[idx], y[idx], self.stump_depth,
                                   n_feats=X.shape[1])  # use all features for stump
            preds_01 = predict_tree(stump, X).astype(int)
            preds_pm = np.where(preds_01 == 1, 1.0, -1.0)

            # Weighted error and alpha
            mis = preds_pm != y_pm
            err = np.dot(w, mis) / w.sum()
            err = np.clip(err, 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - err) / err)

            # Weight update: misclassified → multiply by exp(+α), correct → exp(−α)
            w = w * np.exp(-alpha * y_pm * preds_pm)
            w /= w.sum()   # normalise to probability distribution

            self.alphas.append(alpha)
            self.stumps.append(stump)

        return np.array(weight_history) if track_indices is not None else None

    def predict(self, X):
        agg = np.zeros(len(X))
        for alpha, stump in zip(self.alphas, self.stumps):
            preds_pm = np.where(predict_tree(stump, X).astype(int) == 1, 1.0, -1.0)
            agg += alpha * preds_pm
        return (agg >= 0).astype(int)


# ── Step 1: identify 5 most consistently misclassified training samples ───────
pre_ab = AdaBoost(n_estimators=50, stump_depth=1, random_state=42)
pre_ab.fit(X_train, y_cls_train)   # no tracking

miscount = np.zeros(len(X_train))
for stump in pre_ab.stumps:
    miscount += (predict_tree(stump, X_train).astype(int) != y_cls_train)

top5_idx = np.argsort(miscount)[-5:]
print(f"Top-5 misclassified indices : {top5_idx}")
print(f"Misclassification counts    : {miscount[top5_idx]}")

# ── Step 2: track weights of those 5 over 50 iterations ──────────────────────
ab = AdaBoost(n_estimators=50, stump_depth=1, random_state=42)
weight_history = ab.fit(X_train, y_cls_train, track_indices=top5_idx)
# weight_history shape: (50, 5)

val_acc_ab  = accuracy(y_cls_val,  ab.predict(X_val))
test_acc_ab = accuracy(y_cls_test, ab.predict(X_test))
print(f"AdaBoost — Val Accuracy : {val_acc_ab:.4f}")
print(f"AdaBoost — Test Accuracy: {test_acc_ab:.4f}")

In [ ]:
# ── Plot: exponential weight growth ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
iters = np.arange(1, 51)
for i, idx in enumerate(top5_idx):
    ax.plot(iters, weight_history[:, i], marker='o', ms=3, label=f'Sample {idx}')
ax.set_yscale('log')
ax.set_xlabel('Boosting Iteration')
ax.set_ylabel('Sample Weight (log scale)')
ax.set_title('AdaBoost: Weight Growth of 5 Most Misclassified Samples')
ax.legend()
plt.tight_layout()
plt.show()

### AdaBoost Update Rule — Mathematical Explanation

At iteration $t$, given sample weights $\{w_i^{(t)}\}$:

1. **Train** weak learner $h_t$ on the weighted distribution.
2. **Compute weighted error:**
$$\epsilon_t = \frac{\sum_i w_i^{(t)} \,\mathbf{1}[h_t(x_i) \neq y_i]}{\sum_i w_i^{(t)}}$$
3. **Learner weight:**
$$\alpha_t = \frac{1}{2}\ln\!\left(\frac{1-\epsilon_t}{\epsilon_t}\right)$$
4. **Weight update (with {−1,+1} labels):**
$$w_i^{(t+1)} = w_i^{(t)} \cdot\exp\bigl(-\alpha_t\,y_i\,h_t(x_i)\bigr), \quad\text{then normalise}$$

For a **misclassified** sample $y_i h_t(x_i) = -1$, the factor is $\exp(+\alpha_t) > 1$,  
so its weight **grows exponentially** across rounds of consistent misclassification — exactly the explosion visible in the plot.

---
## Q1.4 — Gradient Boosting Residuals (Regression)

Revert to continuous PM2.5 target.  
Sequentially fit shallow regression trees on pseudo-residuals $(y - F(x))$.  
Plot the variance of training residuals after stages **1, 5, 10, 50**.

In [ ]:
class GradientBoostingRegressor:
    """
    Gradient Boosting with MSE loss.
    Pseudo-residuals = −∂L/∂F = y − F(x)  (negative gradient of ½(y−F)²).
    Each stage fits a shallow regression tree to pseudo-residuals.
    """
    def __init__(self, n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42):
        self.n_estimators  = n_estimators
        self.max_depth     = max_depth
        self.learning_rate = learning_rate
        self.random_state  = random_state
        self.trees = []
        self.F0    = None

    def fit(self, X, y, record_at=(1, 5, 10, 50)):
        """
        Fit GBR. Records residual variance at each stage in `record_at`.
        Returns dict {stage: residual_variance}.
        """
        np.random.seed(self.random_state)
        self.F0 = np.mean(y)                   # initial prediction: global mean
        F = np.full(len(y), self.F0, dtype=float)
        self.trees = []
        variance_at = {}

        for t in range(1, self.n_estimators + 1):
            residuals = y - F                   # pseudo-residuals
            tree = build_tree_reg(X, residuals, self.max_depth)
            update = predict_tree(tree, X)
            F += self.learning_rate * update    # additive update
            self.trees.append(tree)

            if t in record_at:
                variance_at[t] = float(np.var(y - F))
            print(f"  stage {t}/{self.n_estimators}", end='\r', flush=True)

        print()
        return variance_at

    def predict(self, X):
        F = np.full(len(X), self.F0, dtype=float)
        for tree in self.trees:
            F += self.learning_rate * predict_tree(tree, X)
        return F


gbr = GradientBoostingRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42)
RECORD_STAGES = (1, 5, 10, 50)
variance_at = gbr.fit(X_train, y_reg_train, record_at=RECORD_STAGES)

print("Residual variance after each stage:")
for stage, var in variance_at.items():
    print(f"  Stage {stage:2d}: {var:.4f}")

val_mse  = np.mean((y_reg_val  - gbr.predict(X_val)) ** 2)
test_mse = np.mean((y_reg_test - gbr.predict(X_test)) ** 2)
print(f"\nGBR — Val  MSE: {val_mse:.4f}")
print(f"GBR — Test MSE: {test_mse:.4f}")

In [ ]:
# ── Plot: residual variance vs. boosting stage ────────────────────────────────
stages    = list(variance_at.keys())
variances = list(variance_at.values())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(stages, variances, marker='o', lw=2, ms=8, color='steelblue')
for s, v in zip(stages, variances):
    ax.annotate(f'{v:.1f}', (s, v), textcoords='offset points', xytext=(5, 5), fontsize=9)
ax.set_xlabel('Boosting Stage')
ax.set_ylabel('Residual Variance')
ax.set_title('Gradient Boosting: Training Residual Variance vs. Stage')
ax.set_xticks(stages)
plt.tight_layout()
plt.show()